In [ ]:
import glob

data_dir = "/nfs/tcdynasty/tang/tcradar/"

nc_files = sorted(glob.glob(f"{data_dir}/*.nc"))

print("Available files:")
for f in nc_files:
    print(f)


In [ ]:
import xarray as xr

base_dir = "/nfs/tcdynasty/tang/tcradar/"

merge_file = base_dir + "tc_radar_v3m_1997_2019_xy_rel_merge_ships.nc"
swath_file = base_dir + "tc_radar_v3m_1997_2019_xy_rel_swath_ships.nc"

ds_merge = xr.open_dataset(merge_file)
ds_swath = xr.open_dataset(swath_file)

print("MERGE dataset:")
print(ds_merge)

print("\nSWATH dataset:")
print(ds_swath)


In [ ]:
[var for var in ds_merge.data_vars if "storm" in var.lower()
 or "name" in var.lower()
 or "year" in var.lower()
 or "basin" in var.lower()]


In [ ]:
[var for var in ds_merge.coords]


In [ ]:
ds_merge["storm_name"].isel(num_cases=slice(0,10))


In [ ]:
import numpy as np

storm_names = np.char.upper(ds_merge["storm_name"].values.astype(str))
years = ds_merge["merge_year"].values

eduard_idx = np.where(
    (storm_names == "EDOUARD") & (years == 2014)
)[0]

eduard_idx


In [ ]:
ds_edouard = ds_merge.isel(num_cases=[114, 115, 116, 117, 118, 119])

print(ds_edouard)


In [ ]:
import numpy as np

print("Storm:", np.unique(ds_edouard["storm_name"].values))
print("Year:", np.unique(ds_edouard["merge_year"].values))
print("Number of cases:", ds_edouard.dims["num_cases"])


In [ ]:
##----------------------- Tagential Wind Stuff  ----------------

In [ ]:
print(list(ds_swath.data_vars))

In [ ]:
# VARIABLE USING : recentered_tangential_wind

In [ ]:
# ====================================
# USER INPUT
# ====================================
case = 350   # change case here

# ====================================
# IMPORTS
# ====================================
import numpy as np
import matplotlib.pyplot as plt

# ====================================
# EXTRACT DATA
# ====================================
vt = ds_swath['recentered_tangential_wind'].values[case]

east = ds_swath['eastward_distance'].values
north = ds_swath['northward_distance'].values
height = ds_swath['height'].values

# ====================================
# COMPUTE RADIUS GRID
# ====================================
X, Y = np.meshgrid(east, north)
radius = np.sqrt(X**2 + Y**2)

# ====================================
# CREATE RADIAL BINS
# ====================================
r_max = 200
dr = 5

r_bins = np.arange(0, r_max + dr, dr)
r_centers = (r_bins[:-1] + r_bins[1:]) / 2

# ====================================
# AZIMUTHAL MEAN TANGENTIAL WIND
# ====================================
vt_rz = np.zeros((len(height), len(r_centers)))

for k in range(len(height)):
    
    slice2d = vt[:, :, k]
    
    for i in range(len(r_centers)):
        
        mask = (radius >= r_bins[i]) & (radius < r_bins[i+1])
        
        if np.sum(mask) > 10:
            vt_rz[k, i] = np.nanmean(slice2d[mask])
        else:
            vt_rz[k, i] = np.nan

# ====================================
# PLOT
# ====================================
plt.figure(figsize=(8,6))

plt.contourf(
    r_centers,
    height,
    vt_rz,
    levels=np.arange(-10,60,2),
    cmap='plasma'
)

plt.xlabel("Radius (km)")
plt.ylabel("Height (km)")
plt.title("Tangential Wind Structure")

plt.colorbar(label="Tangential Wind (m/s)")

plt.ylim(0,15)
plt.xlim(0,200)

plt.show()

In [ ]:
# ====================================
# USER INPUT
# ====================================

case = 350
# Index number of the radar swath case to analyze.
# Each "case" represents one aircraft radar composite of a tropical cyclone.

r_max = 200
# Maximum radius (km) used when computing azimuthal averages.
# Structure will be calculated from storm center out to 200 km.

dr = 5
# Width of radial bins (km).
# Each bin represents a ring around the storm center.


# ====================================
# IMPORTS
# ====================================

import numpy as np
import matplotlib.pyplot as plt


# ====================================
# EXTRACT DATA FROM THE SWATH DATASET
# ====================================

refl = ds_swath['recentered_reflectivity'].values[case]

# 3D reflectivity field for the selected case.
# Units: dBZ
# Dimensions: [north, east, height]
# Reflectivity values are recentered relative to the storm center.


vt = ds_swath['recentered_tangential_wind'].values[case]

# Tangential wind speed relative to the storm center.
# Units: m/s
# Positive values indicate cyclonic rotation.
# Dimensions: [north, east, height]


east = ds_swath['eastward_distance'].values

# Horizontal distance from storm center in the east-west direction.
# Units: kilometers
# Range typically spans roughly -300 km to +300 km.


north = ds_swath['northward_distance'].values

# Horizontal distance from storm center in the north-south direction.
# Units: kilometers


height = ds_swath['height'].values

# Vertical coordinate of the radar data.
# Units: kilometers above sea level
# Example: 0 km → surface, ~15 km → upper troposphere


# ====================================
# CREATE CARTESIAN GRID RELATIVE TO TC CENTER
# ====================================

X, Y = np.meshgrid(east, north)

# X and Y create a 2D coordinate grid representing the position
# of every radar pixel relative to the storm center.

# X = eastward distance grid
# Y = northward distance grid


radius = np.sqrt(X**2 + Y**2)

# Distance of every grid point from the storm center.
# Computed using the Pythagorean distance formula.

# radius = sqrt(x^2 + y^2)

# Units: km


# ====================================
# CREATE RADIAL BINS FOR AZIMUTHAL AVERAGING
# ====================================

r_bins = np.arange(0, r_max + dr, dr)

# Radial bin edges
# Example: 0 km, 5 km, 10 km, ... 200 km


r_centers = (r_bins[:-1] + r_bins[1:]) / 2

# Midpoint of each radial bin
# These values are used as the x-axis in the radius-height plots.


# ====================================
# INITIALIZE ARRAYS FOR AZIMUTHAL MEANS
# ====================================

refl_rz = np.full((len(height), len(r_centers)), np.nan)

# Reflectivity averaged around each radial ring.
# Dimensions: [height, radius]


vt_rz = np.full((len(height), len(r_centers)), np.nan)

# Tangential wind averaged around each radial ring.
# Dimensions: [height, radius]


# ====================================
# COMPUTE AZIMUTHAL MEAN STRUCTURE
# ====================================

for k in range(len(height)):

    refl_slice = refl[:, :, k]
    # 2D reflectivity field at height level k

    vt_slice = vt[:, :, k]
    # 2D tangential wind field at height level k

    for i in range(len(r_centers)):

        mask = (radius >= r_bins[i]) & (radius < r_bins[i+1])

        # Mask selects all grid points located within the
        # current radial bin.

        if np.sum(mask) > 10:

            refl_rz[k,i] = np.nanmean(refl_slice[mask])

            # Azimuthal mean reflectivity for this radius and height

            vt_rz[k,i] = np.nanmean(vt_slice[mask])

            # Azimuthal mean tangential wind for this radius and height


# ====================================
# VORTEX TILT CALCULATION
# ====================================

centers_x = []
centers_y = []

# These arrays will store the location of the reflectivity centroid
# at each height level.


for k in range(len(height)):

    slice2d = refl[:, :, k]

    # Reflectivity field at height level k

    mask = slice2d >= 10

    # Only include reflectivity values above 10 dBZ.
    # This isolates precipitation structures that represent the vortex.


    if np.sum(mask) < 10:

        centers_x.append(np.nan)
        centers_y.append(np.nan)

        # Skip levels with too little radar signal
        continue


    weights = slice2d * mask

    # Use reflectivity as weighting so stronger convection
    # influences the centroid more strongly.


    x_center = np.nansum(X * weights) / np.nansum(weights)

    # Weighted mean eastward position of the reflectivity field.


    y_center = np.nansum(Y * weights) / np.nansum(weights)

    # Weighted mean northward position of the reflectivity field.


    centers_x.append(x_center)
    centers_y.append(y_center)


centers_x = np.array(centers_x)
centers_y = np.array(centers_y)


# ====================================
# DEFINE TILT RELATIVE TO LOWEST VALID LEVEL
# ====================================

valid = np.where(~np.isnan(centers_x) & ~np.isnan(centers_y))[0]

# Find height levels where centroid calculation succeeded.


if len(valid) > 0:

    ref = valid[0]

    centers_x = centers_x - centers_x[ref]
    centers_y = centers_y - centers_y[ref]

    # Shift all centroid locations so the lowest valid level
    # becomes the reference center (0 km displacement).


tilt_mag = np.sqrt(centers_x**2 + centers_y**2)

# Magnitude of vortex tilt at each height.

# tilt magnitude = sqrt(dx^2 + dy^2)

# Units: kilometers


# ====================================
# CREATE 3-PANEL STRUCTURE FIGURE
# ====================================

fig, axes = plt.subplots(1,3,figsize=(18,6))


# ------------------------------------
# PANEL 1: REFLECTIVITY STRUCTURE
# ------------------------------------

cf1 = axes[0].contourf(
    r_centers,
    height,
    refl_rz,
    levels=np.arange(0,60,2),
    cmap='turbo'
)

# Reflectivity plotted as radius vs height.


axes[0].set_title("Reflectivity Structure")
axes[0].set_xlabel("Radius (km)")
axes[0].set_ylabel("Height (km)")
axes[0].set_xlim(0,r_max)
axes[0].set_ylim(0,15)

fig.colorbar(cf1, ax=axes[0], label="dBZ")


# ------------------------------------
# PANEL 2: TANGENTIAL WIND STRUCTURE
# ------------------------------------

cf2 = axes[1].contourf(
    r_centers,
    height,
    vt_rz,
    levels=np.arange(-10,70,2),
    cmap='plasma'
)

# Tangential wind plotted as radius vs height.
# This reveals eyewall structure and maximum winds.


axes[1].set_title("Tangential Wind Structure")
axes[1].set_xlabel("Radius (km)")
axes[1].set_ylabel("Height (km)")
axes[1].set_xlim(0,r_max)
axes[1].set_ylim(0,15)

fig.colorbar(cf2, ax=axes[1], label="m/s")


# ------------------------------------
# PANEL 3: VORTEX TILT PROFILE
# ------------------------------------

axes[2].plot(tilt_mag, height, linewidth=3)

# Horizontal displacement of vortex center vs height.


axes[2].set_title("Vortex Tilt Profile")
axes[2].set_xlabel("Horizontal Displacement (km)")
axes[2].set_ylabel("Height (km)")
axes[2].grid(True)
axes[2].set_ylim(0,15)


plt.tight_layout()
plt.show()

In [ ]:
# ======================================================
# USER INPUT
# ======================================================
case   = 351   # Radar swath case index to analyze
r_max  = 200   # Maximum radius (km) for azimuthal averaging
dr     = 5     # Radial bin width (km)


# ======================================================
# IMPORTS
# ======================================================
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


# ======================================================
# EXTRACT RADAR DATA FOR SELECTED CASE
# ======================================================
refl   = ds_swath['recentered_reflectivity'].values[case]       # Reflectivity (dBZ)
vt     = ds_swath['recentered_tangential_wind'].values[case]   # Tangential wind (m/s)

east   = ds_swath['eastward_distance'].values                   # Eastward distance (km)
north  = ds_swath['northward_distance'].values                  # Northward distance (km)
height = ds_swath['height'].values                               # Height levels (km)


# ======================================================
# GET CASE TIME FROM SHIPS METADATA
# ======================================================
# Note: This is the SHIPS analysis time (0-hr)
year  = int(ds_swath['year_ships'].values[case, 0])
month = int(ds_swath['month_ships'].values[case, 0])
day   = int(ds_swath['day_ships'].values[case, 0])
hour  = int(ds_swath['hour_ships'].values[case, 0])

storm_time = pd.Timestamp(year, month, day, hour)
time_string = storm_time.strftime("%Y-%m-%d %HZ")


# ======================================================
# CREATE CARTESIAN GRID RELATIVE TO STORM CENTER
# ======================================================
X, Y  = np.meshgrid(east, north)
radius = np.sqrt(X**2 + Y**2)  # Distance from center for each grid point


# ======================================================
# CREATE RADIAL BINS FOR AZIMUTHAL AVERAGES
# ======================================================
r_bins    = np.arange(0, r_max + dr, dr)
r_centers = (r_bins[:-1] + r_bins[1:]) / 2


# ======================================================
# INITIALIZE ARRAYS FOR RADIAL PROFILES
# ======================================================
refl_rz = np.full((len(height), len(r_centers)), np.nan)
vt_rz   = np.full((len(height), len(r_centers)), np.nan)


# ======================================================
# COMPUTE AZIMUTHAL MEANS
# ======================================================
for k in range(len(height)):

    refl_slice = refl[:, :, k]
    vt_slice   = vt[:, :, k]

    for i in range(len(r_centers)):

        mask = (radius >= r_bins[i]) & (radius < r_bins[i+1])

        if np.sum(mask) > 10:
            refl_rz[k, i] = np.nanmean(refl_slice[mask])
            vt_rz[k, i]   = np.nanmean(vt_slice[mask])


# ======================================================
# COMPUTE VORTEX CENTROIDS (FOR TILT CALCULATION)
# ======================================================
centers_x = []
centers_y = []

for k in range(len(height)):

    slice2d = refl[:, :, k]
    mask    = slice2d >= 10  # Consider pixels ≥ 10 dBZ

    if np.sum(mask) < 10:
        centers_x.append(np.nan)
        centers_y.append(np.nan)
        continue

    weights = slice2d * mask
    x_center = np.nansum(X * weights) / np.nansum(weights)
    y_center = np.nansum(Y * weights) / np.nansum(weights)

    centers_x.append(x_center)
    centers_y.append(y_center)

centers_x = np.array(centers_x)
centers_y = np.array(centers_y)


# ======================================================
# DEFINE TILT RELATIVE TO LOWEST VALID LEVEL
# ======================================================
valid_idx = np.where(~np.isnan(centers_x) & ~np.isnan(centers_y))[0]

if len(valid_idx) > 0:
    ref = valid_idx[0]
    centers_x -= centers_x[ref]
    centers_y -= centers_y[ref]

tilt_mag = np.sqrt(centers_x**2 + centers_y**2)


# ======================================================
# PLOT FIGURE: REFLECTIVITY, TANGENTIAL WIND, TILT
# ======================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 6))


# --- PANEL 1: RADIAL REFLECTIVITY ---
cf1 = axes[0].contourf(
    r_centers, height, refl_rz,
    levels=np.arange(0, 60, 2),
    cmap='turbo'
)
axes[0].set_title("Reflectivity Structure")
axes[0].set_xlabel("Radius (km)")
axes[0].set_ylabel("Height (km)")
axes[0].set_xlim(0, r_max)
axes[0].set_ylim(0, 15)
fig.colorbar(cf1, ax=axes[0], label="dBZ")


# --- PANEL 2: RADIAL TANGENTIAL WIND ---
cf2 = axes[1].contourf(
    r_centers, height, vt_rz,
    levels=np.arange(-10, 70, 2),
    cmap='plasma'
)
axes[1].set_title("Tangential Wind Structure")
axes[1].set_xlabel("Radius (km)")
axes[1].set_ylabel("Height (km)")
axes[1].set_xlim(0, r_max)
axes[1].set_ylim(0, 15)
fig.colorbar(cf2, ax=axes[1], label="m/s")


# --- PANEL 3: VORTEX TILT PROFILE ---
axes[2].plot(tilt_mag, height, linewidth=3)
axes[2].set_title("Vortex Tilt Profile")
axes[2].set_xlabel("Horizontal Displacement (km)")
axes[2].set_ylabel("Height (km)")
axes[2].set_ylim(0, 15)
axes[2].grid(True)


# ======================================================
# GLOBAL FIGURE TITLE
# ======================================================
fig.suptitle(
    f"Radar Structure Analysis | Case {case} | {time_string}",
    fontsize=16
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
# ======================================================
# USER INPUT
# ======================================================
case = 351
max_radius = 200   # km along cross-section
n_points   = 400


# ======================================================
# IMPORTS
# ======================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator
import pandas as pd


# ======================================================
# GET UTC TIME
# ======================================================
year  = int(ds_swath['year_ships'].values[case,0])
month = int(ds_swath['month_ships'].values[case,0])
day   = int(ds_swath['day_ships'].values[case,0])
hour  = int(ds_swath['hour_ships'].values[case,0])

utc_string = f"{year:04d}-{month:02d}-{day:02d} {hour:02d}Z"


# ======================================================
# GRID INFORMATION
# ======================================================
east   = ds_swath['eastward_distance'].values
north  = ds_swath['northward_distance'].values
height = ds_swath['height'].values


# ======================================================
# STORM CENTER LOCATION
# ======================================================
storm_lat = ds_swath['lat_ships'].values[case,0]
storm_lon = -abs(ds_swath['lon_ships'].values[case,0])


# ======================================================
# CONVERT KM GRID TO LAT/LON
# ======================================================
deg_per_km_lat = 1/111.0
deg_per_km_lon = 1/(111.0*np.cos(np.deg2rad(storm_lat)))

lon_grid = storm_lon + east*deg_per_km_lon
lat_grid = storm_lat + north*deg_per_km_lat
lon2d, lat2d = np.meshgrid(lon_grid, lat_grid)


# ======================================================
# GET SHEAR VECTOR
# ======================================================
shear_dir = ds_swath['sddc_ships'].values[case,0]
theta = np.deg2rad(270 - shear_dir)
ux = np.cos(theta)
uy = np.sin(theta)


# ======================================================
# PLAN VIEW REFLECTIVITY (1 KM)
# ======================================================
height_idx = np.argmin(np.abs(height - 1.0))
plan_view = ds_swath['recentered_reflectivity'].values[case,:,:,height_idx]


# ======================================================
# BUILD CROSS SECTIONS: REFLECTIVITY & TANGENTIAL WIND
# ======================================================
s = np.linspace(-max_radius, max_radius, n_points)
x_line = s * ux
y_line = s * uy

# Interpolator template
interp = RegularGridInterpolator(
    (north, east),
    np.zeros((len(north), len(east))),
    bounds_error=False,
    fill_value=np.nan
)

cross_refl = np.zeros((len(height), n_points))
cross_vt   = np.zeros((len(height), n_points))

for k in range(len(height)):
    refl_slice = ds_swath['recentered_reflectivity'].values[case,:,:,k]
    vt_slice   = ds_swath['recentered_tangential_wind'].values[case,:,:,k]
    
    interp.values = refl_slice
    points = np.column_stack((y_line, x_line))
    cross_refl[k,:] = interp(points)
    
    interp.values = vt_slice
    cross_vt[k,:] = interp(points)


# ======================================================
# CONVERT CROSS-SECTION LINE TO LAT/LON
# ======================================================
line_lon = storm_lon + x_line * deg_per_km_lon
line_lat = storm_lat + y_line * deg_per_km_lat


# ======================================================
# PLOT FIGURE
# ======================================================
fig, axes = plt.subplots(1, 3, figsize=(18,5))


# ---------------------------
# PANEL 1: Plan View
# ---------------------------
im1 = axes[0].imshow(
    plan_view,
    origin='lower',
    extent=[lon_grid[0], lon_grid[-1], lat_grid[0], lat_grid[-1]],
    cmap='turbo',
    vmin=0, vmax=60
)
axes[0].plot(line_lon, line_lat, color='black', linewidth=2)
axes[0].scatter(storm_lon, storm_lat, color='red', s=60)
axes[0].arrow(
    storm_lon,
    storm_lat,
    ux * 50 * deg_per_km_lon,
    uy * 50 * deg_per_km_lat,
    color='white',
    width=0.05
)
axes[0].set_title("Plan View Reflectivity (1 km)")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")
fig.colorbar(im1, ax=axes[0], label="dBZ")


# ---------------------------
# PANEL 2: Vertical Cross Section (Reflectivity)
# ---------------------------
im2 = axes[1].pcolormesh(
    s, height, cross_refl,
    shading="auto",
    cmap="turbo",
    vmin=0, vmax=60
)
axes[1].set_title("Cross Section Reflectivity")
axes[1].set_xlabel("distance along the cross-section line")
axes[1].set_ylabel("Height (km)")
axes[1].grid(alpha=0.3, linestyle="--")
fig.colorbar(im2, ax=axes[1], label="dBZ")


# ---------------------------
# PANEL 3: Vertical Cross Section (Tangential Wind)
# ---------------------------
im3 = axes[2].pcolormesh(
    s, height, cross_vt,
    shading="auto",
    cmap="plasma",
    vmin=-10, vmax=70
)
axes[2].set_title("Cross Section Tangential Wind")
axes[2].set_xlabel("distance along the cross-section line")
axes[2].set_ylabel("Height (km)")
axes[2].grid(alpha=0.3, linestyle="--")
fig.colorbar(im3, ax=axes[2], label="m/s")


# ---------------------------
# OVERALL FIGURE TITLE
# ---------------------------
fig.suptitle(
    f"Aircraft Cross Section | Case {case} | {utc_string} UTC",
    fontsize=16
)

plt.tight_layout(rect=[0,0.03,1,0.95])
plt.show()

In [ ]:
# ======================================================
# USER INPUT
# ======================================================
case = 351
max_radius = 200   # km along cross-section
n_points   = 400


# ======================================================
# IMPORTS
# ======================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator
import pandas as pd


# ======================================================
# GET SWATH TIMES (RADAR DATA)
# ======================================================

# Swath center time (median of radar pass)
center_time = pd.Timestamp(
    int(ds_swath['swath_year'].values[case]),
    int(ds_swath['swath_month'].values[case]),
    int(ds_swath['swath_day'].values[case]),
    int(ds_swath['swath_hour'].values[case]),
    int(ds_swath['swath_min'].values[case])
)

# Swath start time
start_time = pd.Timestamp(
    int(ds_swath['beg_swath_year'].values[case]),
    int(ds_swath['beg_swath_month'].values[case]),
    int(ds_swath['beg_swath_day'].values[case]),
    int(ds_swath['beg_swath_hour'].values[case]),
    int(ds_swath['beg_swath_min'].values[case])
)

# Swath end time
end_time = pd.Timestamp(
    int(ds_swath['end_swath_year'].values[case]),
    int(ds_swath['end_swath_month'].values[case]),
    int(ds_swath['end_swath_day'].values[case]),
    int(ds_swath['end_swath_hour'].values[case]),
    int(ds_swath['end_swath_min'].values[case])
)

# Lightning window
lightning_start = start_time - pd.Timedelta(minutes=15)
lightning_end   = end_time + pd.Timedelta(minutes=15)

# String for plot title
utc_string = center_time.strftime("%Y-%m-%d %H:%M UTC")

print("Swath Start :", start_time)
print("Swath Center:", center_time)
print("Swath End   :", end_time)
print("Lightning Window:", lightning_start, "to", lightning_end)


# ======================================================
# GRID INFORMATION
# ======================================================
east   = ds_swath['eastward_distance'].values
north  = ds_swath['northward_distance'].values
height = ds_swath['height'].values


# ======================================================
# STORM CENTER LOCATION
# ======================================================
storm_lat = ds_swath['lat_ships'].values[case,0]
storm_lon = -abs(ds_swath['lon_ships'].values[case,0])


# ======================================================
# CONVERT KM GRID TO LAT/LON
# ======================================================
deg_per_km_lat = 1/111.0
deg_per_km_lon = 1/(111.0*np.cos(np.deg2rad(storm_lat)))

lon_grid = storm_lon + east*deg_per_km_lon
lat_grid = storm_lat + north*deg_per_km_lat
lon2d, lat2d = np.meshgrid(lon_grid, lat_grid)


# ======================================================
# GET SHEAR VECTOR
# ======================================================
shear_dir = ds_swath['sddc_ships'].values[case,0]
theta = np.deg2rad(270 - shear_dir)

ux = np.cos(theta)
uy = np.sin(theta)


# ======================================================
# PLAN VIEW REFLECTIVITY (1 KM)
# ======================================================
height_idx = np.argmin(np.abs(height - 1.0))
plan_view = ds_swath['recentered_reflectivity'].values[case,:,:,height_idx]


# ======================================================
# BUILD CROSS-SECTION LINE
# ======================================================
s = np.linspace(-max_radius, max_radius, n_points)

x_line = s * ux
y_line = s * uy


# ======================================================
# INTERPOLATION SETUP
# ======================================================
interp = RegularGridInterpolator(
    (north, east),
    np.zeros((len(north), len(east))),
    bounds_error=False,
    fill_value=np.nan
)

cross_refl = np.zeros((len(height), n_points))
cross_vt   = np.zeros((len(height), n_points))


# ======================================================
# BUILD CROSS SECTIONS
# ======================================================
for k in range(len(height)):

    refl_slice = ds_swath['recentered_reflectivity'].values[case,:,:,k]
    vt_slice   = ds_swath['recentered_tangential_wind'].values[case,:,:,k]

    interp.values = refl_slice
    points = np.column_stack((y_line, x_line))
    cross_refl[k,:] = interp(points)

    interp.values = vt_slice
    cross_vt[k,:] = interp(points)


# ======================================================
# CONVERT CROSS-SECTION LINE TO LAT/LON
# ======================================================
line_lon = storm_lon + x_line * deg_per_km_lon
line_lat = storm_lat + y_line * deg_per_km_lat


# ======================================================
# PLOT FIGURE
# ======================================================
fig, axes = plt.subplots(1, 3, figsize=(18,5))


# ---------------------------
# PANEL 1: PLAN VIEW
# ---------------------------
im1 = axes[0].imshow(
    plan_view,
    origin='lower',
    extent=[lon_grid[0], lon_grid[-1], lat_grid[0], lat_grid[-1]],
    cmap='turbo',
    vmin=0,
    vmax=60
)

axes[0].plot(line_lon, line_lat, color='black', linewidth=2)
axes[0].scatter(storm_lon, storm_lat, color='red', s=60)

axes[0].arrow(
    storm_lon,
    storm_lat,
    ux * 50 * deg_per_km_lon,
    uy * 50 * deg_per_km_lat,
    color='white',
    width=0.05
)

axes[0].set_title("Plan View Reflectivity (1 km)")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")

fig.colorbar(im1, ax=axes[0], label="dBZ")


# ---------------------------
# PANEL 2: CROSS SECTION REFLECTIVITY
# ---------------------------
im2 = axes[1].pcolormesh(
    s,
    height,
    cross_refl,
    shading="auto",
    cmap="turbo",
    vmin=0,
    vmax=60
)

axes[1].set_title("Cross Section Reflectivity")
axes[1].set_xlabel("Distance Along Cross Section (km)")
axes[1].set_ylabel("Height (km)")
axes[1].grid(alpha=0.3, linestyle="--")

fig.colorbar(im2, ax=axes[1], label="dBZ")


# ---------------------------
# PANEL 3: CROSS SECTION TANGENTIAL WIND
# ---------------------------
im3 = axes[2].pcolormesh(
    s,
    height,
    cross_vt,
    shading="auto",
    cmap="plasma",
    vmin=-10,
    vmax=70
)

axes[2].set_title("Cross Section Tangential Wind")
axes[2].set_xlabel("Distance Along Cross Section (km)")
axes[2].set_ylabel("Height (km)")
axes[2].grid(alpha=0.3, linestyle="--")

fig.colorbar(im3, ax=axes[2], label="m/s")


# ======================================================
# FIGURE TITLE
# ======================================================
fig.suptitle(
    f"Aircraft Cross Section | Case {case} | {utc_string}",
    fontsize=16
)

plt.tight_layout(rect=[0,0.03,1,0.95])
plt.show()

In [ ]:
# ======================================================
# USER INPUT
# ======================================================
case = 350
max_radius = 200   # km along cross-section
n_points   = 400

cross_section_deg = 160   # direction of cross section (degrees)


# ======================================================
# IMPORTS
# ======================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator
import pandas as pd


# ======================================================
# GET SWATH TIMES (RADAR DATA)
# ======================================================

center_time = pd.Timestamp(
    int(ds_swath['swath_year'].values[case]),
    int(ds_swath['swath_month'].values[case]),
    int(ds_swath['swath_day'].values[case]),
    int(ds_swath['swath_hour'].values[case]),
    int(ds_swath['swath_min'].values[case])
)

start_time = pd.Timestamp(
    int(ds_swath['beg_swath_year'].values[case]),
    int(ds_swath['beg_swath_month'].values[case]),
    int(ds_swath['beg_swath_day'].values[case]),
    int(ds_swath['beg_swath_hour'].values[case]),
    int(ds_swath['beg_swath_min'].values[case])
)

end_time = pd.Timestamp(
    int(ds_swath['end_swath_year'].values[case]),
    int(ds_swath['end_swath_month'].values[case]),
    int(ds_swath['end_swath_day'].values[case]),
    int(ds_swath['end_swath_hour'].values[case]),
    int(ds_swath['end_swath_min'].values[case])
)

lightning_start = start_time - pd.Timedelta(minutes=15)
lightning_end   = end_time + pd.Timedelta(minutes=15)

utc_string = center_time.strftime("%Y-%m-%d %H:%M UTC")

print("Swath Start :", start_time)
print("Swath Center:", center_time)
print("Swath End   :", end_time)
print("Lightning Window:", lightning_start, "to", lightning_end)


# ======================================================
# GRID INFORMATION
# ======================================================
east   = ds_swath['eastward_distance'].values
north  = ds_swath['northward_distance'].values
height = ds_swath['height'].values


# ======================================================
# STORM CENTER LOCATION
# ======================================================
storm_lat = ds_swath['lat_ships'].values[case,0]
storm_lon = -abs(ds_swath['lon_ships'].values[case,0])


# ======================================================
# CONVERT KM GRID TO LAT/LON
# ======================================================
deg_per_km_lat = 1/111.0
deg_per_km_lon = 1/(111.0*np.cos(np.deg2rad(storm_lat)))

lon_grid = storm_lon + east * deg_per_km_lon
lat_grid = storm_lat + north * deg_per_km_lat
lon2d, lat2d = np.meshgrid(lon_grid, lat_grid)


# ======================================================
# CROSS SECTION DIRECTION
# ======================================================
theta = np.deg2rad(cross_section_deg)

ux = np.cos(theta)
uy = np.sin(theta)


# ======================================================
# PLAN VIEW REFLECTIVITY (1 KM)
# ======================================================
height_idx = np.argmin(np.abs(height - 1.0))
plan_view = ds_swath['recentered_reflectivity'].values[case,:,:,height_idx]


# ======================================================
# BUILD CROSS SECTION LINE
# ======================================================
s = np.linspace(-max_radius, max_radius, n_points)

x_line = s * ux
y_line = s * uy


# ======================================================
# INTERPOLATION SETUP
# ======================================================
interp = RegularGridInterpolator(
    (north, east),
    np.zeros((len(north), len(east))),
    bounds_error=False,
    fill_value=np.nan
)

cross_refl = np.zeros((len(height), n_points))
cross_vt   = np.zeros((len(height), n_points))


# ======================================================
# BUILD CROSS SECTIONS
# ======================================================
for k in range(len(height)):

    refl_slice = ds_swath['recentered_reflectivity'].values[case,:,:,k]
    vt_slice   = ds_swath['recentered_tangential_wind'].values[case,:,:,k]

    interp.values = refl_slice
    points = np.column_stack((y_line, x_line))
    cross_refl[k,:] = interp(points)

    interp.values = vt_slice
    cross_vt[k,:] = interp(points)


# ======================================================
# CONVERT CROSS SECTION LINE TO LAT/LON
# ======================================================
line_lon = storm_lon + x_line * deg_per_km_lon
line_lat = storm_lat + y_line * deg_per_km_lat


# ======================================================
# PLOT FIGURE
# ======================================================
fig, axes = plt.subplots(1, 3, figsize=(18,5))


# ---------------------------
# PLAN VIEW
# ---------------------------
im1 = axes[0].imshow(
    plan_view,
    origin='lower',
    extent=[lon_grid[0], lon_grid[-1], lat_grid[0], lat_grid[-1]],
    cmap='turbo',
    vmin=0,
    vmax=60
)

axes[0].plot(line_lon, line_lat, color='black', linewidth=2)
axes[0].scatter(storm_lon, storm_lat, color='red', s=60)

axes[0].set_title("Plan View Reflectivity (1 km)")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")

fig.colorbar(im1, ax=axes[0], label="dBZ")


# ---------------------------
# CROSS SECTION REFLECTIVITY
# ---------------------------
im2 = axes[1].pcolormesh(
    s,
    height,
    cross_refl,
    shading="auto",
    cmap="turbo",
    vmin=0,
    vmax=60
)

axes[1].set_title("Cross Section Reflectivity")
axes[1].set_xlabel("Distance Along Cross Section (km)")
axes[1].set_ylabel("Height (km)")
axes[1].grid(alpha=0.3, linestyle="--")

fig.colorbar(im2, ax=axes[1], label="dBZ")


# ---------------------------
# CROSS SECTION TANGENTIAL WIND
# ---------------------------
im3 = axes[2].pcolormesh(
    s,
    height,
    cross_vt,
    shading="auto",
    cmap="plasma",
    vmin=-10,
    vmax=70
)

axes[2].set_title("Cross Section Tangential Wind")
axes[2].set_xlabel("Distance Along Cross Section (km)")
axes[2].set_ylabel("Height (km)")
axes[2].grid(alpha=0.3, linestyle="--")

fig.colorbar(im3, ax=axes[2], label="m/s")


# ======================================================
# FIGURE TITLE
# ======================================================
fig.suptitle(
    f"Aircraft Cross Section | Case {case} | {utc_string} | Angle {cross_section_deg}°",
    fontsize=16
)

plt.tight_layout(rect=[0,0.03,1,0.95])
plt.show()

In [ ]:
import glob
import pandas as pd

# --- Load all WWLLN CSV files ---
lightning_files = sorted(
    glob.glob("/nfs/home11/ugrad/2024/es348865/honors_research/250523/WWLLN_201409*.csv")
)

# Concatenate into one DataFrame
df_light_all = pd.concat([pd.read_csv(f) for f in lightning_files])

# Fix longitude to -180 → 180
df_light_all["longitude"] = ((df_light_all["longitude"] + 180) % 360) - 180

# Make datetime column
df_light_all["datetime"] = pd.to_datetime(df_light_all["datetime"], format="%Y/%m/%d %H:%M:%S.%f")

In [ ]:
# ======================================================
# USER INPUT
# ======================================================
case = 351
max_radius = 200       # km along cross-section
n_points   = 400
cross_dir_deg = 180    # cross-section direction (meteorological FROM)
lightning_window_min = 30  # minutes before/after swath center

# ======================================================
# IMPORTS
# ======================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator
import pandas as pd

# ======================================================
# GET SWATH TIMES (RADAR DATA)
# ======================================================
center_time = pd.Timestamp(
    int(ds_swath['swath_year'].values[case]),
    int(ds_swath['swath_month'].values[case]),
    int(ds_swath['swath_day'].values[case]),
    int(ds_swath['swath_hour'].values[case]),
    int(ds_swath['swath_min'].values[case])
)

start_time = pd.Timestamp(
    int(ds_swath['beg_swath_year'].values[case]),
    int(ds_swath['beg_swath_month'].values[case]),
    int(ds_swath['beg_swath_day'].values[case]),
    int(ds_swath['beg_swath_hour'].values[case]),
    int(ds_swath['beg_swath_min'].values[case])
)

end_time = pd.Timestamp(
    int(ds_swath['end_swath_year'].values[case]),
    int(ds_swath['end_swath_month'].values[case]),
    int(ds_swath['end_swath_day'].values[case]),
    int(ds_swath['end_swath_hour'].values[case]),
    int(ds_swath['end_swath_min'].values[case])
)

# Lightning window
# Lightning window: 30 minutes before swath start to 30 minutes after swath end
lightning_start = start_time - pd.Timedelta(minutes=30)
lightning_end   = end_time + pd.Timedelta(minutes=30)

utc_string = center_time.strftime("%Y-%m-%d %H:%M UTC")

print("Swath Start :", start_time)
print("Swath Center:", center_time)
print("Swath End   :", end_time)
print("Lightning Window:", lightning_start, "to", lightning_end)

# ======================================================
# GRID INFORMATION
# ======================================================
east   = ds_swath['eastward_distance'].values
north  = ds_swath['northward_distance'].values
height = ds_swath['height'].values

# ======================================================
# STORM CENTER LOCATION
# ======================================================
storm_lat = ds_swath['lat_ships'].values[case,0]
storm_lon = -abs(ds_swath['lon_ships'].values[case,0])

deg_per_km_lat = 1/111.0
deg_per_km_lon = 1/(111.0*np.cos(np.deg2rad(storm_lat)))

lon_grid = storm_lon + east*deg_per_km_lon
lat_grid = storm_lat + north*deg_per_km_lat
lon2d, lat2d = np.meshgrid(lon_grid, lat_grid)

# ======================================================
# USER-DEFINED CROSS SECTION DIRECTION
# ======================================================
theta = np.deg2rad(270 - cross_dir_deg)
ux = np.cos(theta)
uy = np.sin(theta)

s = np.linspace(-max_radius, max_radius, n_points)
x_line = s * ux
y_line = s * uy

# ======================================================
# PLAN VIEW REFLECTIVITY (1 KM)
# ======================================================
height_idx = np.argmin(np.abs(height - 1.0))
plan_view = ds_swath['recentered_reflectivity'].values[case,:,:,height_idx]

# ======================================================
# BUILD CROSS SECTIONS: REFLECTIVITY & TANGENTIAL WIND
# ======================================================
interp = RegularGridInterpolator(
    (north, east),
    np.zeros((len(north), len(east))),
    bounds_error=False,
    fill_value=np.nan
)

cross_refl = np.zeros((len(height), n_points))
cross_vt   = np.zeros((len(height), n_points))

for k in range(len(height)):
    refl_slice = ds_swath['recentered_reflectivity'].values[case,:,:,k]
    vt_slice   = ds_swath['recentered_tangential_wind'].values[case,:,:,k]
    
    interp.values = refl_slice
    cross_refl[k,:] = interp(np.column_stack((y_line, x_line)))
    
    interp.values = vt_slice
    cross_vt[k,:] = interp(np.column_stack((y_line, x_line)))

# ======================================================
# CONVERT CROSS-SECTION LINE TO LAT/LON
# ======================================================
line_lon = storm_lon + x_line * deg_per_km_lon
line_lat = storm_lat + y_line * deg_per_km_lat

# ======================================================
# 🔥 LIGHTNING FILTERING (WWLLN)
# ======================================================
df_light_sub = df_light_all[
    (df_light_all["datetime"] >= lightning_start) &
    (df_light_all["datetime"] <= lightning_end)
].copy()

df_light_sub["x_km"] = (df_light_sub["longitude"] - storm_lon) / deg_per_km_lon
df_light_sub["y_km"] = (df_light_sub["latitude"] - storm_lat) / deg_per_km_lat

# Keep lightning within plotting radius
df_light_sub = df_light_sub[
    (df_light_sub["x_km"]**2 + df_light_sub["y_km"]**2) <= max_radius**2
]

print(f"Lightning strikes in ±{lightning_window_min} min: {len(df_light_sub)}")

# ======================================================
# PLOT FIGURE
# ======================================================
fig, axes = plt.subplots(1, 3, figsize=(18,5))

# ---------------------------
# PANEL 1: PLAN VIEW
# ---------------------------
im1 = axes[0].imshow(
    plan_view,
    origin='lower',
    extent=[east[0], east[-1], north[0], north[-1]],
    cmap='turbo',
    vmin=0, vmax=60
)

axes[0].plot(x_line, y_line, color='black', linewidth=2)
# Lightning overlay
axes[0].scatter(
    df_light_sub["x_km"],
    df_light_sub["y_km"],
    s=10,
    color="magenta",
    alpha=0.7,
    label="WWLLN"
)
axes[0].set_title("Plan View Reflectivity + Lightning")
axes[0].set_xlabel("Eastward Distance (km)")
axes[0].set_ylabel("Northward Distance (km)")
axes[0].legend()
fig.colorbar(im1, ax=axes[0], label="dBZ")

# ---------------------------
# PANEL 2: CROSS SECTION REFLECTIVITY
# ---------------------------
im2 = axes[1].pcolormesh(
    s, height, cross_refl,
    shading="auto",
    cmap="turbo",
    vmin=0, vmax=60
)
axes[1].set_title(f"Cross Section Reflectivity (FROM {cross_dir_deg}°)")
axes[1].set_xlabel("Distance Along Cross Section (km)")
axes[1].set_ylabel("Height (km)")
axes[1].grid(alpha=0.3, linestyle="--")
fig.colorbar(im2, ax=axes[1], label="dBZ")

# ---------------------------
# PANEL 3: CROSS SECTION TANGENTIAL WIND
# ---------------------------
im3 = axes[2].pcolormesh(
    s, height, cross_vt,
    shading="auto",
    cmap="plasma",
    vmin=-10, vmax=70
)
axes[2].set_title("Cross Section Tangential Wind")
axes[2].set_xlabel("Distance Along Cross Section (km)")
axes[2].set_ylabel("Height (km)")
axes[2].grid(alpha=0.3, linestyle="--")
fig.colorbar(im3, ax=axes[2], label="m/s")

# ---------------------------
# FIGURE TITLE
# ---------------------------
fig.suptitle(
    f"Aircraft Cross Section | Case {case} | {utc_string}",
    fontsize=16
)

plt.tight_layout(rect=[0,0.03,1,0.95])
plt.show()

In [ ]:
# ======================================================
# USER INPUT
# ======================================================
case = 352
max_radius = 200       # km along cross-section
n_points   = 400
cross_dir_deg = 100    # cross-section direction (meteorological FROM)
lightning_window_min = 60  # minutes before swath start / after swath end

# ======================================================
# IMPORTS
# ======================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator
import pandas as pd

# ======================================================
# GET SWATH TIMES (RADAR DATA)
# ======================================================
center_time = pd.Timestamp(
    int(ds_swath['swath_year'].values[case]),
    int(ds_swath['swath_month'].values[case]),
    int(ds_swath['swath_day'].values[case]),
    int(ds_swath['swath_hour'].values[case]),
    int(ds_swath['swath_min'].values[case])
)

start_time = pd.Timestamp(
    int(ds_swath['beg_swath_year'].values[case]),
    int(ds_swath['beg_swath_month'].values[case]),
    int(ds_swath['beg_swath_day'].values[case]),
    int(ds_swath['beg_swath_hour'].values[case]),
    int(ds_swath['beg_swath_min'].values[case])
)

end_time = pd.Timestamp(
    int(ds_swath['end_swath_year'].values[case]),
    int(ds_swath['end_swath_month'].values[case]),
    int(ds_swath['end_swath_day'].values[case]),
    int(ds_swath['end_swath_hour'].values[case]),
    int(ds_swath['end_swath_min'].values[case])
)

# Lightning window: 30 minutes before swath start to 30 minutes after swath end
lightning_start = start_time - pd.Timedelta(minutes=lightning_window_min)
lightning_end   = end_time + pd.Timedelta(minutes=lightning_window_min)

utc_string = center_time.strftime("%Y-%m-%d %H:%M UTC")

print("Swath Start :", start_time)
print("Swath Center:", center_time)
print("Swath End   :", end_time)
print("Lightning Window:", lightning_start, "to", lightning_end)

# ======================================================
# GRID INFORMATION
# ======================================================
east   = ds_swath['eastward_distance'].values
north  = ds_swath['northward_distance'].values
height = ds_swath['height'].values

# ======================================================
# STORM CENTER LOCATION
# ======================================================
storm_lat = ds_swath['lat_ships'].values[case,0]
storm_lon = -abs(ds_swath['lon_ships'].values[case,0])

deg_per_km_lat = 1/111.0
deg_per_km_lon = 1/(111.0*np.cos(np.deg2rad(storm_lat)))

lon_grid = storm_lon + east*deg_per_km_lon
lat_grid = storm_lat + north*deg_per_km_lat
lon2d, lat2d = np.meshgrid(lon_grid, lat_grid)

# ======================================================
# USER-DEFINED CROSS SECTION DIRECTION
# ======================================================
theta = np.deg2rad(270 - cross_dir_deg)
ux = np.cos(theta)
uy = np.sin(theta)

s = np.linspace(-max_radius, max_radius, n_points)
x_line = s * ux
y_line = s * uy

# ======================================================
# PLAN VIEW REFLECTIVITY (1 KM)
# ======================================================
height_idx = np.argmin(np.abs(height - 1.0))
plan_view = ds_swath['recentered_reflectivity'].values[case,:,:,height_idx]

# ======================================================
# BUILD CROSS SECTIONS: REFLECTIVITY & TANGENTIAL WIND
# ======================================================
interp = RegularGridInterpolator(
    (north, east),
    np.zeros((len(north), len(east))),
    bounds_error=False,
    fill_value=np.nan
)

cross_refl = np.zeros((len(height), n_points))
cross_vt   = np.zeros((len(height), n_points))

for k in range(len(height)):
    refl_slice = ds_swath['recentered_reflectivity'].values[case,:,:,k]
    vt_slice   = ds_swath['recentered_tangential_wind'].values[case,:,:,k]
    
    interp.values = refl_slice
    cross_refl[k,:] = interp(np.column_stack((y_line, x_line)))
    
    interp.values = vt_slice
    cross_vt[k,:] = interp(np.column_stack((y_line, x_line)))

# ======================================================
# CONVERT CROSS-SECTION LINE TO LAT/LON
# ======================================================
line_lon = storm_lon + x_line * deg_per_km_lon
line_lat = storm_lat + y_line * deg_per_km_lat

# ======================================================
# LIGHTNING FILTERING (WWLLN)
# ======================================================
df_light_sub = df_light_all[
    (df_light_all["datetime"] >= lightning_start) &
    (df_light_all["datetime"] <= lightning_end)
].copy()

df_light_sub["x_km"] = (df_light_sub["longitude"] - storm_lon) / deg_per_km_lon
df_light_sub["y_km"] = (df_light_sub["latitude"] - storm_lat) / deg_per_km_lat

# Keep lightning within plotting radius
df_light_sub = df_light_sub[
    (df_light_sub["x_km"]**2 + df_light_sub["y_km"]**2) <= max_radius**2
]

print(f"Lightning strikes in window: {len(df_light_sub)}")

# ======================================================
# PLOT FIGURE
# ======================================================
fig, axes = plt.subplots(1, 3, figsize=(18,5))

# ---------------------------
# PANEL 1: PLAN VIEW
# ---------------------------
im1 = axes[0].imshow(
    plan_view,
    origin='lower',
    extent=[east[0], east[-1], north[0], north[-1]],
    cmap='turbo',
    vmin=0, vmax=60
)

axes[0].plot(x_line, y_line, color='black', linewidth=2)
# Lightning overlay
axes[0].scatter(
    df_light_sub["x_km"],
    df_light_sub["y_km"],
    s=10,
    color="magenta",
    alpha=0.7,
    label="WWLLN"
)
axes[0].set_title("Plan View Reflectivity + Lightning")
axes[0].set_xlabel("Eastward Distance (km)")
axes[0].set_ylabel("Northward Distance (km)")
axes[0].legend()
fig.colorbar(im1, ax=axes[0], label="dBZ")

# ---------------------------
# PANEL 2: CROSS SECTION REFLECTIVITY
# ---------------------------
im2 = axes[1].pcolormesh(
    s, height, cross_refl,
    shading="auto",
    cmap="turbo",
    vmin=0, vmax=60
)
axes[1].set_title(f"Cross Section Reflectivity (FROM {cross_dir_deg}°)")
axes[1].set_xlabel("Distance Along Cross Section (km)")
axes[1].set_ylabel("Height (km)")
axes[1].grid(alpha=0.3, linestyle="--")
fig.colorbar(im2, ax=axes[1], label="dBZ")

# ---------------------------
# PANEL 3: CROSS SECTION TANGENTIAL WIND
# ---------------------------
im3 = axes[2].pcolormesh(
    s, height, cross_vt,
    shading="auto",
    cmap="plasma",
    vmin=-10, vmax=70
)
axes[2].set_title("Cross Section Tangential Wind")
axes[2].set_xlabel("Distance Along Cross Section (km)")
axes[2].set_ylabel("Height (km)")
axes[2].grid(alpha=0.3, linestyle="--")
fig.colorbar(im3, ax=axes[2], label="m/s")

# ---------------------------
# FIGURE TITLE
# ---------------------------
fig.suptitle(
    f"Aircraft Cross Section | Case {case} | {utc_string}",
    fontsize=16
)

plt.tight_layout(rect=[0,0.03,1,0.95])
plt.show()

In [ ]:
# ======================================================
# USER INPUT
# ======================================================
case = 357
max_radius = 200       # km along cross-section
n_points   = 400
cross_dir_deg = 100    # cross-section direction (meteorological FROM)
lightning_window_min = 60  # minutes before swath start / after swath end

# ======================================================
# IMPORTS
# ======================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator
import pandas as pd

# ======================================================
# GET SWATH TIMES (RADAR DATA)
# ======================================================
center_time = pd.Timestamp(
    int(ds_swath['swath_year'].values[case]),
    int(ds_swath['swath_month'].values[case]),
    int(ds_swath['swath_day'].values[case]),
    int(ds_swath['swath_hour'].values[case]),
    int(ds_swath['swath_min'].values[case])
)

start_time = pd.Timestamp(
    int(ds_swath['beg_swath_year'].values[case]),
    int(ds_swath['beg_swath_month'].values[case]),
    int(ds_swath['beg_swath_day'].values[case]),
    int(ds_swath['beg_swath_hour'].values[case]),
    int(ds_swath['beg_swath_min'].values[case])
)

end_time = pd.Timestamp(
    int(ds_swath['end_swath_year'].values[case]),
    int(ds_swath['end_swath_month'].values[case]),
    int(ds_swath['end_swath_day'].values[case]),
    int(ds_swath['end_swath_hour'].values[case]),
    int(ds_swath['end_swath_min'].values[case])
)

# Lightning window: minutes before swath start / after swath end
lightning_start = start_time - pd.Timedelta(minutes=lightning_window_min)
lightning_end   = end_time + pd.Timedelta(minutes=lightning_window_min)

utc_string = center_time.strftime("%Y-%m-%d %H:%M UTC")

print("------ Lightning Check ------")
print("Swath Start :", start_time)
print("Swath Center:", center_time)
print("Swath End   :", end_time)
print("Lightning Window:", lightning_start, "to", lightning_end)

# ======================================================
# GRID INFORMATION
# ======================================================
east   = ds_swath['eastward_distance'].values
north  = ds_swath['northward_distance'].values
height = ds_swath['height'].values

# ======================================================
# STORM CENTER LOCATION (FIXED)
# ======================================================
storm_lat = ds_swath['lat_ships'].values[case,0]
storm_lon = ds_swath['lon_ships'].values[case,0]  # DO NOT FORCE NEGATIVE

deg_per_km_lat = 1/111.0
deg_per_km_lon = 1/(111.0*np.cos(np.deg2rad(storm_lat)))

lon_grid = storm_lon + east*deg_per_km_lon
lat_grid = storm_lat + north*deg_per_km_lat
lon2d, lat2d = np.meshgrid(lon_grid, lat_grid)

# ======================================================
# USER-DEFINED CROSS SECTION DIRECTION
# ======================================================
theta = np.deg2rad(270 - cross_dir_deg)
ux = np.cos(theta)
uy = np.sin(theta)

s = np.linspace(-max_radius, max_radius, n_points)
x_line = s * ux
y_line = s * uy

# ======================================================
# PLAN VIEW REFLECTIVITY (1 KM)
# ======================================================
height_idx = np.argmin(np.abs(height - 1.0))
plan_view = ds_swath['recentered_reflectivity'].values[case,:,:,height_idx]

# ======================================================
# BUILD CROSS SECTIONS: REFLECTIVITY & TANGENTIAL WIND
# ======================================================
interp = RegularGridInterpolator(
    (north, east),
    np.zeros((len(north), len(east))),
    bounds_error=False,
    fill_value=np.nan
)

cross_refl = np.zeros((len(height), n_points))
cross_vt   = np.zeros((len(height), n_points))

for k in range(len(height)):
    refl_slice = ds_swath['recentered_reflectivity'].values[case,:,:,k]
    vt_slice   = ds_swath['recentered_tangential_wind'].values[case,:,:,k]
    
    interp.values = refl_slice
    cross_refl[k,:] = interp(np.column_stack((y_line, x_line)))
    
    interp.values = vt_slice
    cross_vt[k,:] = interp(np.column_stack((y_line, x_line)))

# ======================================================
# CONVERT CROSS-SECTION LINE TO LAT/LON
# ======================================================
line_lon = storm_lon + x_line * deg_per_km_lon
line_lat = storm_lat + y_line * deg_per_km_lat

# ======================================================
# LIGHTNING FILTERING (WWLLN)
# ======================================================
df_light_sub = df_light_all[
    (df_light_all["datetime"] >= lightning_start) &
    (df_light_all["datetime"] <= lightning_end)
].copy()

# Convert to storm-relative km
df_light_sub["x_km"] = (df_light_sub["longitude"] - storm_lon) / deg_per_km_lon
df_light_sub["y_km"] = (df_light_sub["latitude"] - storm_lat) / deg_per_km_lat

# Keep lightning within plotting radius
df_light_sub = df_light_sub[
    (df_light_sub["x_km"]**2 + df_light_sub["y_km"]**2) <= max_radius**2
]

print(f"Lightning strikes in window & within {max_radius} km: {len(df_light_sub)}")
print("Example lightning entries (first 5):")
print(df_light_sub.head())

# ======================================================
# PLOT FIGURE
# ======================================================
fig, axes = plt.subplots(1, 3, figsize=(18,5))

# ---------------------------
# PANEL 1: PLAN VIEW
# ---------------------------
im1 = axes[0].imshow(
    plan_view,
    origin='lower',
    extent=[east[0], east[-1], north[0], north[-1]],
    cmap='turbo',
    vmin=0, vmax=60
)

axes[0].plot(x_line, y_line, color='black', linewidth=2)

# Lightning overlay
axes[0].scatter(
    df_light_sub["x_km"],
    df_light_sub["y_km"],
    s=10,
    color="magenta",
    alpha=0.7,
    label="WWLLN"
)

axes[0].set_title("Plan View Reflectivity + Lightning")
axes[0].set_xlabel("Eastward Distance (km)")
axes[0].set_ylabel("Northward Distance (km)")
axes[0].legend()
fig.colorbar(im1, ax=axes[0], label="dBZ")

# ---------------------------
# PANEL 2: CROSS SECTION REFLECTIVITY
# ---------------------------
im2 = axes[1].pcolormesh(
    s, height, cross_refl,
    shading="auto",
    cmap="turbo",
    vmin=0, vmax=60
)
axes[1].set_title(f"Cross Section Reflectivity (FROM {cross_dir_deg}°)")
axes[1].set_xlabel("Distance Along Cross Section (km)")
axes[1].set_ylabel("Height (km)")
axes[1].grid(alpha=0.3, linestyle="--")
fig.colorbar(im2, ax=axes[1], label="dBZ")

# ---------------------------
# PANEL 3: CROSS SECTION TANGENTIAL WIND
# ---------------------------
im3 = axes[2].pcolormesh(
    s, height, cross_vt,
    shading="auto",
    cmap="plasma",
    vmin=-10, vmax=70
)
axes[2].set_title("Cross Section Tangential Wind")
axes[2].set_xlabel("Distance Along Cross Section (km)")
axes[2].set_ylabel("Height (km)")
axes[2].grid(alpha=0.3, linestyle="--")
fig.colorbar(im3, ax=axes[2], label="m/s")

# ---------------------------
# FIGURE TITLE
# ---------------------------
fig.suptitle(
    f"Aircraft Cross Section | Case {case} | {utc_string}",
    fontsize=16
)

plt.tight_layout(rect=[0,0.03,1,0.95])
plt.show()

In [ ]:
# ======================================================
# USER INPUT
# ======================================================
case = 357
max_radius = 200       # km along cross-section
n_points   = 400
cross_dir_deg = 100    # cross-section direction (meteorological FROM)
lightning_window_min = 60  # minutes before swath start / after swath end

# ======================================================
# IMPORTS
# ======================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator
import pandas as pd

# ======================================================
# GET SWATH TIMES (RADAR DATA)
# ======================================================
center_time = pd.Timestamp(
    int(ds_swath['swath_year'].values[case]),
    int(ds_swath['swath_month'].values[case]),
    int(ds_swath['swath_day'].values[case]),
    int(ds_swath['swath_hour'].values[case]),
    int(ds_swath['swath_min'].values[case])
)

start_time = pd.Timestamp(
    int(ds_swath['beg_swath_year'].values[case]),
    int(ds_swath['beg_swath_month'].values[case]),
    int(ds_swath['beg_swath_day'].values[case]),
    int(ds_swath['beg_swath_hour'].values[case]),
    int(ds_swath['beg_swath_min'].values[case])
)

end_time = pd.Timestamp(
    int(ds_swath['end_swath_year'].values[case]),
    int(ds_swath['end_swath_month'].values[case]),
    int(ds_swath['end_swath_day'].values[case]),
    int(ds_swath['end_swath_hour'].values[case]),
    int(ds_swath['end_swath_min'].values[case])
)

lightning_start = start_time - pd.Timedelta(minutes=lightning_window_min)
lightning_end   = end_time + pd.Timedelta(minutes=lightning_window_min)

utc_string = center_time.strftime("%Y-%m-%d %H:%M UTC")

print("------ Lightning Check ------")
print("Swath Start :", start_time)
print("Swath Center:", center_time)
print("Swath End   :", end_time)
print("Lightning Window:", lightning_start, "to", lightning_end)

# ======================================================
# GRID INFORMATION
# ======================================================
east   = ds_swath['eastward_distance'].values
north  = ds_swath['northward_distance'].values
height = ds_swath['height'].values

# ======================================================
# STORM CENTER LOCATION
# ======================================================
storm_lat = ds_swath['lat_ships'].values[case,0]
storm_lon = ds_swath['lon_ships'].values[case,0]

deg_per_km_lat = 1/111.0
deg_per_km_lon = 1/(111.0*np.cos(np.deg2rad(storm_lat)))

# ======================================================
# CROSS SECTION LINE
# ======================================================
theta = np.deg2rad(270 - cross_dir_deg)
ux = np.cos(theta)
uy = np.sin(theta)

s = np.linspace(-max_radius, max_radius, n_points)
x_line = s * ux
y_line = s * uy

# ---- NEW: Define endpoints A and B ----
xA, yA = x_line[0], y_line[0]
xB, yB = x_line[-1], y_line[-1]

# ======================================================
# PLAN VIEW REFLECTIVITY
# ======================================================
height_idx = np.argmin(np.abs(height - 1.0))
plan_view = ds_swath['recentered_reflectivity'].values[case,:,:,height_idx]

# ======================================================
# CROSS SECTIONS
# ======================================================
interp = RegularGridInterpolator(
    (north, east),
    np.zeros((len(north), len(east))),
    bounds_error=False,
    fill_value=np.nan
)

cross_refl = np.zeros((len(height), n_points))
cross_vt   = np.zeros((len(height), n_points))

for k in range(len(height)):
    refl_slice = ds_swath['recentered_reflectivity'].values[case,:,:,k]
    vt_slice   = ds_swath['recentered_tangential_wind'].values[case,:,:,k]
    
    interp.values = refl_slice
    cross_refl[k,:] = interp(np.column_stack((y_line, x_line)))
    
    interp.values = vt_slice
    cross_vt[k,:] = interp(np.column_stack((y_line, x_line)))

# ======================================================
# LIGHTNING FILTERING
# ======================================================
df_light_sub = df_light_all[
    (df_light_all["datetime"] >= lightning_start) &
    (df_light_all["datetime"] <= lightning_end)
].copy()

df_light_sub["x_km"] = (df_light_sub["longitude"] - storm_lon) / deg_per_km_lon
df_light_sub["y_km"] = (df_light_sub["latitude"] - storm_lat) / deg_per_km_lat

df_light_sub = df_light_sub[
    (df_light_sub["x_km"]**2 + df_light_sub["y_km"]**2) <= max_radius**2
]

print(f"Lightning strikes in window & within {max_radius} km: {len(df_light_sub)}")

# ======================================================
# PLOT FIGURE
# ======================================================
fig, axes = plt.subplots(1, 3, figsize=(18,5))

# ---------------------------
# PANEL 1: PLAN VIEW
# ---------------------------
im1 = axes[0].imshow(
    plan_view,
    origin='lower',
    extent=[east[0], east[-1], north[0], north[-1]],
    cmap='turbo',
    vmin=0, vmax=60
)

# Cross-section line
axes[0].plot(x_line, y_line, color='black', linewidth=2)

# ---- NEW: A/B markers ----
axes[0].scatter([xA, xB], [yA, yB], color='black', s=50, zorder=5)

axes[0].text(xA, yA, 'A', fontsize=12, fontweight='bold',
             ha='right', va='bottom')

axes[0].text(xB, yB, 'B', fontsize=12, fontweight='bold',
             ha='left', va='top')

# Lightning
axes[0].scatter(
    df_light_sub["x_km"],
    df_light_sub["y_km"],
    s=10,
    color="magenta",
    alpha=0.7,
    label="WWLLN"
)

axes[0].set_title("Plan View Reflectivity + Lightning")
axes[0].set_xlabel("Eastward Distance (km)")
axes[0].set_ylabel("Northward Distance (km)")
axes[0].legend()
fig.colorbar(im1, ax=axes[0], label="dBZ")

# ---------------------------
# PANEL 2: CROSS SECTION REFLECTIVITY
# ---------------------------
im2 = axes[1].pcolormesh(
    s, height, cross_refl,
    shading="auto",
    cmap="turbo",
    vmin=0, vmax=60
)

# ---- NEW: A/B labels ----
axes[1].text(s[0], height[-1], 'A', fontsize=14, fontweight='bold',
             ha='center', va='top')
axes[1].text(s[-1], height[-1], 'B', fontsize=14, fontweight='bold',
             ha='center', va='top')

# Optional guide lines
axes[1].axvline(s[0], linestyle='--', alpha=0.5)
axes[1].axvline(s[-1], linestyle='--', alpha=0.5)

axes[1].set_title(f"Cross Section Reflectivity (FROM {cross_dir_deg}°)")
axes[1].set_xlabel("Distance Along Cross Section (km)")
axes[1].set_ylabel("Height (km)")
axes[1].grid(alpha=0.3, linestyle="--")
fig.colorbar(im2, ax=axes[1], label="dBZ")

# ---------------------------
# PANEL 3: CROSS SECTION TANGENTIAL WIND
# ---------------------------
im3 = axes[2].pcolormesh(
    s, height, cross_vt,
    shading="auto",
    cmap="plasma",
    vmin=-10, vmax=70
)

# ---- NEW: A/B labels ----
axes[2].text(s[0], height[-1], 'A', fontsize=14, fontweight='bold',
             ha='center', va='top')
axes[2].text(s[-1], height[-1], 'B', fontsize=14, fontweight='bold',
             ha='center', va='top')

# Optional guide lines
axes[2].axvline(s[0], linestyle='--', alpha=0.5)
axes[2].axvline(s[-1], linestyle='--', alpha=0.5)

axes[2].set_title("Cross Section Tangential Wind")
axes[2].set_xlabel("Distance Along Cross Section (km)")
axes[2].set_ylabel("Height (km)")
axes[2].grid(alpha=0.3, linestyle="--")
fig.colorbar(im3, ax=axes[2], label="m/s")

# ---------------------------
# TITLE
# ---------------------------
fig.suptitle(
    f"Aircraft Cross Section | Case {case} | {utc_string}",
    fontsize=16
)

plt.tight_layout(rect=[0,0.03,1,0.95])
plt.show()

In [ ]:
# ======================================================
# USER INPUT
# ======================================================
case = 354
max_radius = 200       # km along cross-section
n_points   = 400
cross_dir_deg = 260    # cross-section direction (meteorological FROM)
lightning_window_min = 60  # minutes before swath start / after swath end

# ======================================================
# IMPORTS
# ======================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator
import pandas as pd

# ======================================================
# GET SWATH TIMES (RADAR DATA)
# ======================================================
center_time = pd.Timestamp(
    int(ds_swath['swath_year'].values[case]),
    int(ds_swath['swath_month'].values[case]),
    int(ds_swath['swath_day'].values[case]),
    int(ds_swath['swath_hour'].values[case]),
    int(ds_swath['swath_min'].values[case])
)

start_time = pd.Timestamp(
    int(ds_swath['beg_swath_year'].values[case]),
    int(ds_swath['beg_swath_month'].values[case]),
    int(ds_swath['beg_swath_day'].values[case]),
    int(ds_swath['beg_swath_hour'].values[case]),
    int(ds_swath['beg_swath_min'].values[case])
)

end_time = pd.Timestamp(
    int(ds_swath['end_swath_year'].values[case]),
    int(ds_swath['end_swath_month'].values[case]),
    int(ds_swath['end_swath_day'].values[case]),
    int(ds_swath['end_swath_hour'].values[case]),
    int(ds_swath['end_swath_min'].values[case])
)

lightning_start = start_time - pd.Timedelta(minutes=lightning_window_min)
lightning_end   = end_time + pd.Timedelta(minutes=lightning_window_min)

utc_string = center_time.strftime("%Y-%m-%d %H:%M UTC")


# ======================================================
# GRID INFORMATION
# ======================================================
east   = ds_swath['eastward_distance'].values
north  = ds_swath['northward_distance'].values
height = ds_swath['height'].values

# ======================================================
# STORM CENTER LOCATION
# ======================================================
storm_lat = ds_swath['lat_ships'].values[case, 0]
storm_lon = ds_swath['lon_ships'].values[case, 0]

# Fix longitude convention: SHIPS stores Western Atlantic as positive,
# but WWLLN uses -180 to 180, so negate
storm_lon = -abs(storm_lon)

print(f"Storm center: lat={storm_lat:.2f}, lon={storm_lon:.2f}")

deg_per_km_lat = 1/111.0
deg_per_km_lon = 1/(111.0*np.cos(np.deg2rad(storm_lat)))

# ======================================================
# CROSS SECTION LINE
# ======================================================
theta = np.deg2rad(270 - cross_dir_deg)
ux = np.cos(theta)
uy = np.sin(theta)

s = np.linspace(-max_radius, max_radius, n_points)
x_line = s * ux
y_line = s * uy

# Define endpoints A and B
xA, yA = x_line[0], y_line[0]
xB, yB = x_line[-1], y_line[-1]

# ======================================================
# PLAN VIEW REFLECTIVITY
# ======================================================
height_idx = np.argmin(np.abs(height - 1.0))
plan_view = ds_swath['recentered_reflectivity'].values[case,:,:,height_idx]

# ======================================================
# CROSS SECTIONS
# ======================================================
interp = RegularGridInterpolator(
    (north, east),
    np.zeros((len(north), len(east))),
    bounds_error=False,
    fill_value=np.nan
)

cross_refl = np.zeros((len(height), n_points))
cross_vt   = np.zeros((len(height), n_points))

for k in range(len(height)):
    refl_slice = ds_swath['recentered_reflectivity'].values[case,:,:,k]
    vt_slice   = ds_swath['recentered_tangential_wind'].values[case,:,:,k]
    
    interp.values = refl_slice
    cross_refl[k,:] = interp(np.column_stack((y_line, x_line)))
    
    interp.values = vt_slice
    cross_vt[k,:] = interp(np.column_stack((y_line, x_line)))


# ======================================================
# PLOT FIGURE
# ======================================================
fig, axes = plt.subplots(1, 3, figsize=(18,5))

# ---------------------------
# PANEL 1: PLAN VIEW
# ---------------------------
im1 = axes[0].imshow(
    plan_view,
    origin='lower',
    extent=[east[0], east[-1], north[0], north[-1]],
    cmap='turbo',
    vmin=0, vmax=60
)

# Cross-section line
axes[0].plot(x_line, y_line, color='black', linewidth=2)

# A/B markers
axes[0].scatter([xA, xB], [yA, yB], color='black', s=50, zorder=5)
axes[0].text(xA, yA, 'A', fontsize=12, fontweight='bold', ha='right', va='bottom')
axes[0].text(xB, yB, 'B', fontsize=12, fontweight='bold', ha='left',  va='top')


axes[0].set_title("Plan View Reflectivity + Lightning")
axes[0].set_xlabel("Eastward Distance (km)")
axes[0].set_ylabel("Northward Distance (km)")
axes[0].legend()
fig.colorbar(im1, ax=axes[0], label="dBZ")

# ---------------------------
# PANEL 2: CROSS SECTION REFLECTIVITY
# ---------------------------
im2 = axes[1].pcolormesh(
    s, height, cross_refl,
    shading="auto",
    cmap="turbo",
    vmin=0, vmax=60
)

axes[1].text(s[0],  height[-1], 'A', fontsize=14, fontweight='bold', ha='center', va='top')
axes[1].text(s[-1], height[-1], 'B', fontsize=14, fontweight='bold', ha='center', va='top')
axes[1].axvline(s[0],  linestyle='--', alpha=0.5)
axes[1].axvline(s[-1], linestyle='--', alpha=0.5)

axes[1].set_title(f"Cross Section Reflectivity (FROM {cross_dir_deg}°)")
axes[1].set_xlabel("Distance Along Cross Section (km)")
axes[1].set_ylabel("Height (km)")
axes[1].grid(alpha=0.3, linestyle="--")
fig.colorbar(im2, ax=axes[1], label="dBZ")

# ---------------------------
# PANEL 3: CROSS SECTION TANGENTIAL WIND
# ---------------------------
im3 = axes[2].pcolormesh(
    s, height, cross_vt,
    shading="auto",
    cmap="plasma",
    vmin=-10, vmax=70
)

axes[2].text(s[0],  height[-1], 'A', fontsize=14, fontweight='bold', ha='center', va='top')
axes[2].text(s[-1], height[-1], 'B', fontsize=14, fontweight='bold', ha='center', va='top')
axes[2].axvline(s[0],  linestyle='--', alpha=0.5)
axes[2].axvline(s[-1], linestyle='--', alpha=0.5)

axes[2].set_title("Cross Section Tangential Wind")
axes[2].set_xlabel("Distance Along Cross Section (km)")
axes[2].set_ylabel("Height (km)")
axes[2].grid(alpha=0.3, linestyle="--")
fig.colorbar(im3, ax=axes[2], label="m/s")

# ---------------------------
# TITLE
# ---------------------------
fig.suptitle(
    f"Aircraft Cross Section | Case {case} | {utc_string}",
    fontsize=16
)

plt.tight_layout(rect=[0,0.03,1,0.95])
plt.show()

In [ ]:
results = []

for c in range(350, 358):
    # Get swath start and end times
    st = pd.Timestamp(
        int(ds_swath['beg_swath_year'].values[c]),
        int(ds_swath['beg_swath_month'].values[c]),
        int(ds_swath['beg_swath_day'].values[c]),
        int(ds_swath['beg_swath_hour'].values[c]),
        int(ds_swath['beg_swath_min'].values[c])
    )
    et = pd.Timestamp(
        int(ds_swath['end_swath_year'].values[c]),
        int(ds_swath['end_swath_month'].values[c]),
        int(ds_swath['end_swath_day'].values[c]),
        int(ds_swath['end_swath_hour'].values[c]),
        int(ds_swath['end_swath_min'].values[c])
    )

    # Storm center
    slat = ds_swath['lat_ships'].values[c, 0]
    slon = -abs(ds_swath['lon_ships'].values[c, 0])

    deg_per_km_lat = 1 / 111.0
    deg_per_km_lon = 1 / (111.0 * np.cos(np.deg2rad(slat)))

    # Filter lightning to swath time window
    df_t = df_light_all[
        (df_light_all["datetime"] >= st) &
        (df_light_all["datetime"] <= et)
    ].copy()

    # Compute distance from storm center
    df_t["r_km"] = np.sqrt(
        ((df_t["longitude"] - slon) / deg_per_km_lon)**2 +
        ((df_t["latitude"]  - slat) / deg_per_km_lat)**2
    )

    n_strikes = (df_t["r_km"] <= 350).sum()

    results.append({
        "case":              c,
        "start_time":        st,
        "end_time":          et,
        "lat":               slat,
        "lon":               slon,
        "n_lightning_350km": n_strikes
    })

df_cases = pd.DataFrame(results)
print(df_cases.to_string(index=False))

In [ ]:
c = 350
st = pd.Timestamp(2014, 9, 14, 14, 41)
et = pd.Timestamp(2014, 9, 14, 15, 24)
slat = 18.3
slon = -42.3

deg_per_km_lat = 1 / 111.0
deg_per_km_lon = 1 / (111.0 * np.cos(np.deg2rad(slat)))

# Step 1: How many rows exist in this time window?
df_t = df_light_all[
    (df_light_all["datetime"] >= st) &
    (df_light_all["datetime"] <= et)
].copy()
print(f"Rows in time window: {len(df_t)}")
print(df_t[["datetime","latitude","longitude"]].head(10))

# Step 2: What is the closest strike?
df_t["r_km"] = np.sqrt(
    ((df_t["longitude"] - slon) / deg_per_km_lon)**2 +
    ((df_t["latitude"]  - slat) / deg_per_km_lat)**2
)
print(f"\nClosest strike: {df_t['r_km'].min():.1f} km")
print(df_t.nsmallest(5, "r_km")[["datetime","latitude","longitude","r_km"]])

# Step 3: Check what IS near 18.3N, -42.3W in the full dataset (no time filter)
df_pos = df_light_all.copy()
df_pos["r_km"] = np.sqrt(
    ((df_pos["longitude"] - slon) / deg_per_km_lon)**2 +
    ((df_pos["latitude"]  - slat) / deg_per_km_lat)**2
)
df_near = df_pos[df_pos["r_km"] <= 350]
print(f"\nStrikes within 350 km of storm (entire Sep dataset): {len(df_near)}")
print(df_near[["datetime","latitude","longitude","r_km"]].head(10))

In [ ]:
results = []

for c in range(350, 358):
    st = pd.Timestamp(
        int(ds_swath['beg_swath_year'].values[c]),
        int(ds_swath['beg_swath_month'].values[c]),
        int(ds_swath['beg_swath_day'].values[c]),
        int(ds_swath['beg_swath_hour'].values[c]),
        int(ds_swath['beg_swath_min'].values[c])
    )
    et = pd.Timestamp(
        int(ds_swath['end_swath_year'].values[c]),
        int(ds_swath['end_swath_month'].values[c]),
        int(ds_swath['end_swath_day'].values[c]),
        int(ds_swath['end_swath_hour'].values[c]),
        int(ds_swath['end_swath_min'].values[c])
    )

    lightning_start = st - pd.Timedelta(hours=1)
    lightning_end   = et + pd.Timedelta(hours=1)

    slat = ds_swath['lat_ships'].values[c, 0]
    slon = -abs(ds_swath['lon_ships'].values[c, 0])

    deg_per_km_lat = 1 / 111.0
    deg_per_km_lon = 1 / (111.0 * np.cos(np.deg2rad(slat)))

    df_t = df_light_all[
        (df_light_all["datetime"] >= lightning_start) &
        (df_light_all["datetime"] <= lightning_end)
    ].copy()

    df_t["r_km"] = np.sqrt(
        ((df_t["longitude"] - slon) / deg_per_km_lon)**2 +
        ((df_t["latitude"]  - slat) / deg_per_km_lat)**2
    )

    n_strikes = (df_t["r_km"] <= 350).sum()

    results.append({
        "case":              c,
        "lightning_start":   lightning_start,
        "lightning_end":     lightning_end,
        "lat":               slat,
        "lon":               slon,
        "n_lightning_350km": n_strikes
    })

df_cases = pd.DataFrame(results)
print(df_cases.to_string(index=False))

In [ ]:
from datetime import datetime as dt

track_data = {
    "datetime": [
        "2014-09-11 00:00", "2014-09-11 06:00", "2014-09-11 12:00",
        "2014-09-11 18:00", "2014-09-12 00:00", "2014-09-12 06:00", "2014-09-12 12:00",
        "2014-09-12 18:00", "2014-09-13 00:00", "2014-09-13 06:00", "2014-09-13 12:00",
        "2014-09-13 18:00", "2014-09-14 00:00", "2014-09-14 06:00", "2014-09-14 12:00",
        "2014-09-14 18:00", "2014-09-15 00:00", "2014-09-15 06:00", "2014-09-15 12:00",
        "2014-09-15 18:00", "2014-09-16 00:00", "2014-09-16 06:00", "2014-09-16 12:00",
        "2014-09-16 18:00", "2014-09-17 00:00", "2014-09-17 06:00", "2014-09-17 12:00",
        "2014-09-17 18:00", "2014-09-18 00:00", "2014-09-18 06:00", "2014-09-18 12:00",
        "2014-09-18 18:00", "2014-09-19 00:00", "2014-09-19 06:00", "2014-09-19 12:00",
        "2014-09-19 18:00", "2014-09-20 00:00", "2014-09-20 06:00", "2014-09-20 12:00",
        "2014-09-20 18:00"
    ],
    "lat": [
        14.2, 14.9, 15.8, 16.5, 17.2, 17.7, 18.3, 18.9, 19.6, 20.4, 21.4, 22.3,
        23.0, 23.7, 24.4, 25.1, 25.8, 26.5, 27.0, 27.7, 28.5, 29.5, 30.6, 31.7,
        32.9, 34.3, 35.7, 37.2, 38.3, 39.3, 39.9, 39.9, 39.9, 39.9, 39.5, 39.5,
        39.4, 39.3, 39.1, 38.8
    ],
    "lon": [
        -34.4, -35.5, -36.5, -37.7, -39.2, -40.8, -42.3, -43.6, -44.8,
        -45.7, -46.4, -47.0, -47.9, -49.0, -50.3, -51.5, -52.7, -54.0, -55.1,
        -56.1, -56.7, -57.3, -57.8, -57.7, -57.0, -56.0, -54.5, -52.4, -49.5,
        -46.7, -44.1, -42.1, -40.1, -38.9, -38.1, -37.4, -36.6, -35.7, -34.4, -33.0
    ],
    "pressure": [
        1009, 1009, 1008, 1007, 1006, 1005, 1000, 999, 998, 997, 996, 995,
        994, 991, 985, 980, 975, 970, 966, 962, 960, 956, 955, 958, 959, 958,
        957, 955, 958, 964, 969, 973, 980, 988, 994, 1001, 1004, 1006, 1008, 1010
    ],
    "wind": [
        25, 25, 30, 30, 35, 35, 40, 40, 40, 45, 45, 50, 55, 60, 70, 75, 80,
        85, 90, 95, 95, 100, 105, 95, 85, 85, 80, 80, 75, 70, 70, 65, 60, 55,
        45, 40, 35, 30, 30, 25
    ],
    "stage": [
        "low", "low", "tropical depression", "tropical depression",
        "tropical storm", "tropical storm", "tropical storm", "tropical storm",
        "tropical storm", "tropical storm", "tropical storm", "tropical storm",
        "tropical storm", "tropical storm", "hurricane", "hurricane", "hurricane",
        "hurricane", "hurricane", "hurricane", "hurricane", "hurricane",
        "hurricane", "hurricane", "hurricane", "hurricane", "hurricane",
        "hurricane", "hurricane", "hurricane", "hurricane", "hurricane",
        "tropical storm", "tropical storm", "tropical storm", "low", "low",
        "low", "low", "low"
    ]
}


In [ ]:
import pandas as pd
import xarray as xr
import glob
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import numpy as np
import os
import datetime as dt
import subprocess
from tropycal import tracks

#########################
# Animation time range
#########################
start_cutoff = pd.Timestamp("2014-09-11 00:00:00")
end_cutoff = pd.Timestamp("2014-09-18 23:59:00")

#########################
# Load storm track data
#########################
df_track = pd.DataFrame(track_data)
df_track["datetime"] = pd.to_datetime(df_track["datetime"])
df_track = df_track.set_index("datetime")

# Interpolate numeric columns every 30 min
interp_times = pd.date_range(
    start=df_track.index.min(), end=df_track.index.max(), freq="30min"
)
numeric_cols = ["lat", "lon", "wind", "pressure"]
track_interp = df_track[numeric_cols].reindex(df_track.index.union(interp_times))
track_interp = track_interp.interpolate(method="time")
track_interp = track_interp.loc[
    (track_interp.index >= start_cutoff) & (track_interp.index <= end_cutoff)
]

# Forward-fill stage column
track_interp["stage"] = df_track["stage"].reindex(track_interp.index).ffill()


In [ ]:
for c in range(350, 358):
    st = pd.Timestamp(
        int(ds_swath['beg_swath_year'].values[c]),
        int(ds_swath['beg_swath_month'].values[c]),
        int(ds_swath['beg_swath_day'].values[c]),
        int(ds_swath['beg_swath_hour'].values[c]),
        int(ds_swath['beg_swath_min'].values[c])
    )
    slat = ds_swath['lat_ships'].values[c, 0]
    slon = -abs(ds_swath['lon_ships'].values[c, 0])

    # What does the track interpolation give for this time?
    track_lat = track_interp["lat"].asof(st)
    track_lon = track_interp["lon"].asof(st)

    print(f"Case {c} | {st}")
    print(f"  ds_swath position : {slat:.2f}N, {slon:.2f}W")
    print(f"  track_interp position: {track_lat:.2f}N, {track_lon:.2f}W")
    print()

In [ ]:
results = []

for c in range(350, 358):
    st = pd.Timestamp(
        int(ds_swath['beg_swath_year'].values[c]),
        int(ds_swath['beg_swath_month'].values[c]),
        int(ds_swath['beg_swath_day'].values[c]),
        int(ds_swath['beg_swath_hour'].values[c]),
        int(ds_swath['beg_swath_min'].values[c])
    )
    et = pd.Timestamp(
        int(ds_swath['end_swath_year'].values[c]),
        int(ds_swath['end_swath_month'].values[c]),
        int(ds_swath['end_swath_day'].values[c]),
        int(ds_swath['end_swath_hour'].values[c]),
        int(ds_swath['end_swath_min'].values[c])
    )

    lightning_start = st - pd.Timedelta(hours=1)
    lightning_end   = et + pd.Timedelta(hours=1)

    # Use track_interp instead of ds_swath for correct storm position
    slat = track_interp["lat"].asof(st)
    slon = track_interp["lon"].asof(st)

    deg_per_km_lat = 1 / 111.0
    deg_per_km_lon = 1 / (111.0 * np.cos(np.deg2rad(slat)))

    df_t = df_light_all[
        (df_light_all["datetime"] >= lightning_start) &
        (df_light_all["datetime"] <= lightning_end)
    ].copy()

    df_t["r_km"] = np.sqrt(
        ((df_t["longitude"] - slon) / deg_per_km_lon)**2 +
        ((df_t["latitude"]  - slat) / deg_per_km_lat)**2
    )

    n_strikes = (df_t["r_km"] <= 350).sum()

    results.append({
        "case":              c,
        "lightning_start":   lightning_start,
        "lightning_end":     lightning_end,
        "lat":               slat,
        "lon":               slon,
        "n_lightning_350km": n_strikes
    })

df_cases = pd.DataFrame(results)
print(df_cases.to_string(index=False))

In [ ]:
# ======================================================
# USER INPUT
# ======================================================
case = 356
max_radius = 200       # km along cross-section
n_points   = 400
cross_dir_deg = 180   # cross-section direction (meteorological FROM)
lightning_window_min = 60  # minutes before swath start / after swath end

# ======================================================
# IMPORTS
# ======================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator
import pandas as pd

# ======================================================
# GET SWATH TIMES (RADAR DATA)
# ======================================================
center_time = pd.Timestamp(
    int(ds_swath['swath_year'].values[case]),
    int(ds_swath['swath_month'].values[case]),
    int(ds_swath['swath_day'].values[case]),
    int(ds_swath['swath_hour'].values[case]),
    int(ds_swath['swath_min'].values[case])
)

start_time = pd.Timestamp(
    int(ds_swath['beg_swath_year'].values[case]),
    int(ds_swath['beg_swath_month'].values[case]),
    int(ds_swath['beg_swath_day'].values[case]),
    int(ds_swath['beg_swath_hour'].values[case]),
    int(ds_swath['beg_swath_min'].values[case])
)

end_time = pd.Timestamp(
    int(ds_swath['end_swath_year'].values[case]),
    int(ds_swath['end_swath_month'].values[case]),
    int(ds_swath['end_swath_day'].values[case]),
    int(ds_swath['end_swath_hour'].values[case]),
    int(ds_swath['end_swath_min'].values[case])
)

lightning_start = start_time - pd.Timedelta(minutes=lightning_window_min)
lightning_end   = end_time + pd.Timedelta(minutes=lightning_window_min)

utc_string = center_time.strftime("%Y-%m-%d %H:%M UTC")

print("------ Lightning Check ------")
print("Swath Start :", start_time)
print("Swath Center:", center_time)
print("Swath End   :", end_time)
print("Lightning Window:", lightning_start, "to", lightning_end)

# ======================================================
# GRID INFORMATION
# ======================================================
east   = ds_swath['eastward_distance'].values
north  = ds_swath['northward_distance'].values
height = ds_swath['height'].values

# ======================================================
# STORM CENTER LOCATION
# Use track_interp (best track) instead of ds_swath SHIPS,
# which has incorrect positions for Edouard cases
# ======================================================
storm_lat = track_interp["lat"].asof(start_time)
storm_lon = track_interp["lon"].asof(start_time)

print(f"Storm center (best track): lat={storm_lat:.3f}, lon={storm_lon:.3f}")

deg_per_km_lat = 1/111.0
deg_per_km_lon = 1/(111.0*np.cos(np.deg2rad(storm_lat)))

# ======================================================
# CROSS SECTION LINE
# ======================================================
theta = np.deg2rad(270 - cross_dir_deg)
ux = np.cos(theta)
uy = np.sin(theta)

s = np.linspace(-max_radius, max_radius, n_points)
x_line = s * ux
y_line = s * uy

# Define endpoints A and B
xA, yA = x_line[0], y_line[0]
xB, yB = x_line[-1], y_line[-1]

# ======================================================
# PLAN VIEW REFLECTIVITY
# ======================================================
height_idx = np.argmin(np.abs(height - 1.0))
plan_view = ds_swath['recentered_reflectivity'].values[case,:,:,height_idx]

# ======================================================
# CROSS SECTIONS
# ======================================================
interp = RegularGridInterpolator(
    (north, east),
    np.zeros((len(north), len(east))),
    bounds_error=False,
    fill_value=np.nan
)

cross_refl = np.zeros((len(height), n_points))
cross_vt   = np.zeros((len(height), n_points))

for k in range(len(height)):
    refl_slice = ds_swath['recentered_reflectivity'].values[case,:,:,k]
    vt_slice   = ds_swath['recentered_tangential_wind'].values[case,:,:,k]

    interp.values = refl_slice
    cross_refl[k,:] = interp(np.column_stack((y_line, x_line)))

    interp.values = vt_slice
    cross_vt[k,:] = interp(np.column_stack((y_line, x_line)))

# ======================================================
# LIGHTNING FILTERING
# ======================================================
df_light_sub = df_light_all[
    (df_light_all["datetime"] >= lightning_start) &
    (df_light_all["datetime"] <= lightning_end)
].copy()

df_light_sub["x_km"] = (df_light_sub["longitude"] - storm_lon) / deg_per_km_lon
df_light_sub["y_km"] = (df_light_sub["latitude"]  - storm_lat) / deg_per_km_lat

df_light_sub = df_light_sub[
    (df_light_sub["x_km"]**2 + df_light_sub["y_km"]**2) <= max_radius**2
]

print(f"Lightning strikes in window & within {max_radius} km: {len(df_light_sub)}")

# ======================================================
# PLOT FIGURE
# ======================================================
fig, axes = plt.subplots(1, 3, figsize=(18,5))

# ---------------------------
# PANEL 1: PLAN VIEW
# ---------------------------
im1 = axes[0].imshow(
    plan_view,
    origin='lower',
    extent=[east[0], east[-1], north[0], north[-1]],
    cmap='turbo',
    vmin=0, vmax=60
)

# Cross-section line
axes[0].plot(x_line, y_line, color='black', linewidth=2)

# A/B markers
axes[0].scatter([xA, xB], [yA, yB], color='black', s=50, zorder=5)
axes[0].text(xA, yA, 'A', fontsize=12, fontweight='bold', ha='right', va='bottom')
axes[0].text(xB, yB, 'B', fontsize=12, fontweight='bold', ha='left',  va='top')

# Lightning
axes[0].scatter(
    df_light_sub["x_km"],
    df_light_sub["y_km"],
    s=10,
    color="magenta",
    alpha=0.7,
    zorder=10,
    label="WWLLN"
)

axes[0].set_title("Plan View Reflectivity + Lightning")
axes[0].set_xlabel("Eastward Distance (km)")
axes[0].set_ylabel("Northward Distance (km)")
axes[0].legend()
fig.colorbar(im1, ax=axes[0], label="dBZ")

# ---------------------------
# PANEL 2: CROSS SECTION REFLECTIVITY
# ---------------------------
im2 = axes[1].pcolormesh(
    s, height, cross_refl,
    shading="auto",
    cmap="turbo",
    vmin=0, vmax=60
)

axes[1].text(s[0],  height[-1], 'A', fontsize=14, fontweight='bold', ha='center', va='top')
axes[1].text(s[-1], height[-1], 'B', fontsize=14, fontweight='bold', ha='center', va='top')
axes[1].axvline(s[0],  linestyle='--', alpha=0.5)
axes[1].axvline(s[-1], linestyle='--', alpha=0.5)

axes[1].set_title(f"Cross Section Reflectivity (FROM {cross_dir_deg}°)")
axes[1].set_xlabel("Distance Along Cross Section (km)")
axes[1].set_ylabel("Height (km)")
axes[1].grid(alpha=0.3, linestyle="--")
fig.colorbar(im2, ax=axes[1], label="dBZ")

# ---------------------------
# PANEL 3: CROSS SECTION TANGENTIAL WIND
# ---------------------------
im3 = axes[2].pcolormesh(
    s, height, cross_vt,
    shading="auto",
    cmap="plasma",
    vmin=-10, vmax=70
)

axes[2].text(s[0],  height[-1], 'A', fontsize=14, fontweight='bold', ha='center', va='top')
axes[2].text(s[-1], height[-1], 'B', fontsize=14, fontweight='bold', ha='center', va='top')
axes[2].axvline(s[0],  linestyle='--', alpha=0.5)
axes[2].axvline(s[-1], linestyle='--', alpha=0.5)

axes[2].set_title("Cross Section Tangential Wind")
axes[2].set_xlabel("Distance Along Cross Section (km)")
axes[2].set_ylabel("Height (km)")
axes[2].grid(alpha=0.3, linestyle="--")
fig.colorbar(im3, ax=axes[2], label="m/s")

# ---------------------------
# TITLE
# ---------------------------
fig.suptitle(
    f"Aircraft Cross Section | Case {case} | {utc_string}",
    fontsize=16
)

plt.tight_layout(rect=[0,0.03,1,0.95])
plt.show()

In [ ]:
# ======================================================
# USER INPUT
# ======================================================
case = 350
max_radius = 200       # km along cross-section
n_points   = 400
cross_dir_deg = 270   # cross-section direction (meteorological FROM)
lightning_window_min = 60  # minutes before swath start / after swath end

# ======================================================
# IMPORTS
# ======================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator
import pandas as pd

# ======================================================
# GET SWATH TIMES (RADAR DATA)
# ======================================================
center_time = pd.Timestamp(
    int(ds_swath['swath_year'].values[case]),
    int(ds_swath['swath_month'].values[case]),
    int(ds_swath['swath_day'].values[case]),
    int(ds_swath['swath_hour'].values[case]),
    int(ds_swath['swath_min'].values[case])
)

start_time = pd.Timestamp(
    int(ds_swath['beg_swath_year'].values[case]),
    int(ds_swath['beg_swath_month'].values[case]),
    int(ds_swath['beg_swath_day'].values[case]),
    int(ds_swath['beg_swath_hour'].values[case]),
    int(ds_swath['beg_swath_min'].values[case])
)

end_time = pd.Timestamp(
    int(ds_swath['end_swath_year'].values[case]),
    int(ds_swath['end_swath_month'].values[case]),
    int(ds_swath['end_swath_day'].values[case]),
    int(ds_swath['end_swath_hour'].values[case]),
    int(ds_swath['end_swath_min'].values[case])
)

lightning_start = start_time - pd.Timedelta(minutes=lightning_window_min)
lightning_end   = end_time + pd.Timedelta(minutes=lightning_window_min)

utc_string = center_time.strftime("%Y-%m-%d %H:%M UTC")

print("------ Lightning Check ------")
print("Swath Start :", start_time)
print("Swath Center:", center_time)
print("Swath End   :", end_time)
print("Lightning Window:", lightning_start, "to", lightning_end)

# ======================================================
# GRID INFORMATION
# ======================================================
east   = ds_swath['eastward_distance'].values
north  = ds_swath['northward_distance'].values
height = ds_swath['height'].values

# ======================================================
# STORM CENTER LOCATION
# Use track_interp (best track) instead of ds_swath SHIPS,
# which has incorrect positions for Edouard cases
# ======================================================
storm_lat = track_interp["lat"].asof(start_time)
storm_lon = track_interp["lon"].asof(start_time)

print(f"Storm center (best track): lat={storm_lat:.3f}, lon={storm_lon:.3f}")

deg_per_km_lat = 1/111.0
deg_per_km_lon = 1/(111.0*np.cos(np.deg2rad(storm_lat)))

# ======================================================
# CROSS SECTION LINE
# ======================================================
theta = np.deg2rad(270 - cross_dir_deg)
ux = np.cos(theta)
uy = np.sin(theta)

s = np.linspace(-max_radius, max_radius, n_points)
x_line = s * ux
y_line = s * uy

# Define endpoints A and B
xA, yA = x_line[0], y_line[0]
xB, yB = x_line[-1], y_line[-1]

# ======================================================
# PLAN VIEW REFLECTIVITY
# ======================================================
height_idx = np.argmin(np.abs(height - 1.0))
plan_view = ds_swath['recentered_reflectivity'].values[case,:,:,height_idx]

# ======================================================
# CROSS SECTIONS
# ======================================================
interp = RegularGridInterpolator(
    (north, east),
    np.zeros((len(north), len(east))),
    bounds_error=False,
    fill_value=np.nan
)

cross_refl = np.zeros((len(height), n_points))
cross_vt   = np.zeros((len(height), n_points))

for k in range(len(height)):
    refl_slice = ds_swath['recentered_reflectivity'].values[case,:,:,k]
    vt_slice   = ds_swath['recentered_tangential_wind'].values[case,:,:,k]

    interp.values = refl_slice
    cross_refl[k,:] = interp(np.column_stack((y_line, x_line)))

    interp.values = vt_slice
    cross_vt[k,:] = interp(np.column_stack((y_line, x_line)))

# ======================================================
# LIGHTNING FILTERING
# ======================================================
df_light_sub = df_light_all[
    (df_light_all["datetime"] >= lightning_start) &
    (df_light_all["datetime"] <= lightning_end)
].copy()

df_light_sub["x_km"] = (df_light_sub["longitude"] - storm_lon) / deg_per_km_lon
df_light_sub["y_km"] = (df_light_sub["latitude"]  - storm_lat) / deg_per_km_lat

df_light_sub = df_light_sub[
    (df_light_sub["x_km"]**2 + df_light_sub["y_km"]**2) <= max_radius**2
]

print(f"Lightning strikes in window & within {max_radius} km: {len(df_light_sub)}")

# ======================================================
# PLOT FIGURE
# ======================================================
fig, axes = plt.subplots(1, 2, figsize=(18,5))

# ---------------------------
# PANEL 1: PLAN VIEW
# ---------------------------
im1 = axes[0].imshow(
    plan_view,
    origin='lower',
    extent=[east[0], east[-1], north[0], north[-1]],
    cmap='turbo',
    vmin=0, vmax=60
)

# Cross-section line
axes[0].plot(x_line, y_line, color='black', linewidth=2)

# A/B markers
axes[0].scatter([xA, xB], [yA, yB], color='black', s=50, zorder=5)
axes[0].text(xA, yA, 'A', fontsize=14, fontweight='bold', ha='right', va='bottom', color='darkred')
axes[0].text(xB, yB, 'B', fontsize=14, fontweight='bold', ha='left',  va='top', color='darkred')

# Lightning
axes[0].scatter(
    df_light_sub["x_km"],
    df_light_sub["y_km"],
    s=10,
    color="magenta",
    alpha=0.7,
    zorder=10,
    label="WWLLN"
)

axes[0].set_title("Plan View Reflectivity + Lightning")
axes[0].set_xlabel("Eastward Distance (km)")
axes[0].set_ylabel("Northward Distance (km)")
axes[0].legend()
fig.colorbar(im1, ax=axes[0], label="dBZ")

# ---------------------------
# PANEL 2: CROSS SECTION REFLECTIVITY
# ---------------------------
im2 = axes[1].pcolormesh(
    s, height, cross_refl,
    shading="auto",
    cmap="turbo",
    vmin=0, vmax=60
)

axes[1].text(s[0],  height[-1], 'A', fontsize=16, fontweight='bold', ha='center', va='top', color='darkred')
axes[1].text(s[-1], height[-1], 'B', fontsize=16, fontweight='bold', ha='center', va='top', color='darkred')
axes[1].axvline(s[0],  linestyle='--', alpha=0.5)
axes[1].axvline(s[-1], linestyle='--', alpha=0.5)

axes[1].set_title(f"Cross Section Reflectivity (FROM {cross_dir_deg}°)")
axes[1].set_xlabel("Distance Along Cross Section (km)")
axes[1].set_ylabel("Height (km)")
axes[1].grid(alpha=0.3, linestyle="--")
fig.colorbar(im2, ax=axes[1], label="dBZ")


# ---------------------------
# TITLE
# ---------------------------
fig.suptitle(
    f"Aircraft Cross Section | Case {case} | {utc_string}",
    fontsize=16
)

plt.tight_layout(rect=[0,0.03,1,0.95])
plt.show()